# Mini-projet : SPARQL et préparation de données pour la recherche
Ce notebook a pour but de montrer comment transformer des données issues d’un graphe de connaissances en un jeu de données clair et réutilisable pour la recherche.

À partir d’informations extraites depuis une source ouverte, le travail consiste à vérifier, nettoyer et organiser les données afin qu’elles puissent être facilement comprises, partagées et exploitées par des chercheurs, ou intégrées plus tard dans une base de connaissances.

## Aperçu du pipeline
<p style="text-align: center;">
  <img src="images/pipeline_overview.png" width="70%">
</p>

## Source des données
<p style="text-align: center;">
  <img src="images/Wikidata-logo.png" width="10%">
</p>

Les données proviennent de Wikidata, un référentiel ouvert et multilingue du web sémantique. Il sert fréquemment de point d’appui pour l’enrichissement, l’alignement et la standardisation des données en humanités numériques.

In [1]:
import requests
import pandas as pd
from pathlib import Path

In [14]:
response.raise_for_status()

In [18]:
# Query path
query_path = Path("queries/authors.rq")
query = query_path.read_text()

# Endpoint wikidata
url = "https://query.wikidata.org/sparql"

# Send request
response = requests.get(
    url,
    params={"format": "json", "query": query},
    headers={"User-Agent": "sparql-demo"}
)

response.raise_for_status()

data = response.json()["results"]["bindings"]
data

[{'author': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q272'},
  'birthPlace': {'type': 'uri',
   'value': 'http://www.wikidata.org/entity/Q270230'},
  'birthDate': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime',
   'type': 'literal',
   'value': '1888-03-13T00:00:00Z'},
  'authorLabel': {'xml:lang': 'fr', 'type': 'literal', 'value': 'Paul Morand'},
  'birthPlaceLabel': {'xml:lang': 'fr',
   'type': 'literal',
   'value': '8e arrondissement de Paris'}},
 {'author': {'type': 'uri', 'value': 'http://www.wikidata.org/entity/Q493'},
  'birthPlace': {'type': 'uri',
   'value': 'http://www.wikidata.org/entity/Q2960786'},
  'birthDate': {'datatype': 'http://www.w3.org/2001/XMLSchema#dateTime',
   'type': 'literal',
   'value': '1854-10-20T00:00:00Z'},
  'authorLabel': {'xml:lang': 'fr',
   'type': 'literal',
   'value': 'Arthur Rimbaud'},
  'birthPlaceLabel': {'xml:lang': 'fr',
   'type': 'literal',
   'value': 'Charleville'}},
 {'author': {'type': 'uri', 'value': 'htt

In [3]:
# Transform into dataframe
rows = []
for item in data:
    rows.append({
        "author": item.get("authorLabel", {}).get("value"),
        "birthDate": item.get("birthDate", {}).get("value"),
        "birthPlace": item.get("birthPlaceLabel", {}).get("value"),
    })

df = pd.DataFrame(rows)
print("DataFrame shape :")
print(df.shape)
df.head()

DataFrame shape :
(100, 3)


,author,birthDate,birthPlace
0,Paul Morand,1888-03-13T00:00:00Z,8e arrondissement de Paris
1,Arthur Rimbaud,1854-10-20T00:00:00Z,Charleville
2,Charles Baudelaire,1821-04-09T00:00:00Z,Paris
3,Charles Baudelaire,1821-04-09T00:00:00Z,ancien 11e arrondissement de Paris
4,Stendhal,1783-01-23T00:00:00Z,Grenoble


In [4]:
# Save file
output_path = Path("data/authors.csv")
output_path.parent.mkdir(exist_ok=True)
df.to_csv(output_path, index=False)

## 2. Data curation step
Opérations principales :

- Analyser et simplifier les informations temporelles
- Nettoyer les champs textuels
- Supprimer les enregistrements incomplets
- Créer des identifiants internes stables

In [5]:
# Normalize dates (keep only the year)
df["birthYear"] = pd.to_datetime(
    df["birthDate"], errors="coerce"
).dt.year
df["birthYear"] = df["birthYear"].astype("Int64")

df.head()

,author,birthDate,birthPlace,birthYear
0,Paul Morand,1888-03-13T00:00:00Z,8e arrondissement de Paris,1888
1,Arthur Rimbaud,1854-10-20T00:00:00Z,Charleville,1854
2,Charles Baudelaire,1821-04-09T00:00:00Z,Paris,1821
3,Charles Baudelaire,1821-04-09T00:00:00Z,ancien 11e arrondissement de Paris,1821
4,Stendhal,1783-01-23T00:00:00Z,Grenoble,1783


In [6]:
# Clean text (remove extra spaces and ensure consistent string formatting)
df["author"] = df["author"].astype(str).str.strip()
df["birthPlace"] = df["birthPlace"].astype(str).str.strip()
print("Rows after cleaning:", len(df))

,author,birthDate,birthPlace,birthYear
0,Paul Morand,1888-03-13T00:00:00Z,8e arrondissement de Paris,1888
1,Arthur Rimbaud,1854-10-20T00:00:00Z,Charleville,1854
2,Charles Baudelaire,1821-04-09T00:00:00Z,Paris,1821
3,Charles Baudelaire,1821-04-09T00:00:00Z,ancien 11e arrondissement de Paris,1821
4,Stendhal,1783-01-23T00:00:00Z,Grenoble,1783


In [7]:
# Remove incomplete records
before = len(df)
df = df.dropna(subset=["author", "birthYear"])
after = len(df)
print("Rows after cleaning:", len(df))

Removed 1 incomplete rows


In [8]:
# Create internal identifiers
df["internal_id"] = range(1, len(df) + 1)

# Select final columns
df = df[["internal_id", "author", "birthYear", "birthPlace"]]
df.head()

,internal_id,author,birthYear,birthPlace
0,1,Paul Morand,1888,8e arrondissement de Paris
1,2,Arthur Rimbaud,1854,Charleville
2,3,Charles Baudelaire,1821,Paris
3,4,Charles Baudelaire,1821,ancien 11e arrondissement de Paris
4,5,Stendhal,1783,Grenoble


In [9]:
# Deduplicate authors
df = df.sort_values("internal_id")
df = df.drop_duplicates(subset=["author", "birthYear"], keep="first")
print("Rows after cleaning:", len(df))

After dedup: 93


In [13]:
# 3. Reset identifiers
df = df.reset_index(drop=True)
df["internal_id"] = range(1, len(df) + 1)
print("DataFrame shape :")
print(df.shape)
df.head()

DataFrame shape :
(93, 4)


,internal_id,author,birthYear,birthPlace
0,1,Paul Morand,1888,8e arrondissement de Paris
1,2,Arthur Rimbaud,1854,Charleville
2,3,Charles Baudelaire,1821,Paris
3,4,Stendhal,1783,Grenoble
4,5,Émile Zola,1840,rue Saint-Joseph
...,...,...,...,...
88,89,Hubert Rouger,1875,Calvisson
89,90,Oscar Wilde,1854,Dublin
90,91,Maurice Alhoy,1802,Paris
91,92,Henry Murger,1822,ancien 2e arrondissement de Paris


In [11]:
# Save file
output_path = Path("data/authors_curated.csv")
output_path.parent.mkdir(exist_ok=True)
df.to_csv(output_path, index=False)